# 04 — Modelo Predictivo (extra, no requerido)

**Objetivo:** predecir si una oportunidad de venta se va a ganar o perder,
usando solo información disponible mientras la oportunidad todavía está
abierta (sin fuga de datos hacia el futuro).

Esto no es parte de los entregables pedidos -- es un extra que se apoya
directamente en el modelo Gold ya construido. Se optó por un modelo simple
e interpretable (regresión logística) en vez de algo más sofisticado,
priorizando poder explicar y justificar cada decisión sobre maximizar
precisión.

**Conexión con un hallazgo ya documentado:** en el notebook de insights
(03_business_insights.ipynb) encontramos que el promedio de actividades es
casi idéntico entre oportunidades ganadas y perdidas (~3.4-3.6). Este
modelo permite confirmar (o refutar) esa observación con más rigor
estadístico, no solo comparando promedios.


In [1]:
import pandas as pd
import numpy as np
import psycopg2
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

def get_connection():
    return psycopg2.connect(
        host=os.environ.get("WAREHOUSE_HOST", "postgres-warehouse"),
        port=os.environ.get("WAREHOUSE_PORT", "5432"),
        dbname=os.environ.get("WAREHOUSE_DB", "warehouse"),
        user=os.environ.get("WAREHOUSE_USER", "rodrick"),
        password=os.environ.get("WAREHOUSE_PASSWORD", "rodrick123"),
    )

conn = get_connection()
print("Conexión OK.")


Conexión OK.


## 1. Preparar el dataset

**Regla clave para evitar fuga de datos (data leakage):** solo se usan
columnas que existían *mientras la oportunidad seguía abierta* -- por
ejemplo, `close_date` queda excluida a propósito, porque su sola presencia
ya delata que la oportunidad se cerró.

Se filtra a oportunidades en estado terminal (`won`/`lost`) -- las que
siguen en pipeline no tienen una respuesta conocida todavía, así que no
sirven para entrenar ni evaluar.


In [2]:
query = """
SELECT
    o.opportunity_id,
    o.amount,
    o.stage,
    a.industry,
    a.country,
    a.annual_revenue,
    a.employees,
    COUNT(act.activity_id) AS activity_count
FROM gold.fact_opportunities o
JOIN gold.dim_account a ON a.sk_account = o.sk_account
LEFT JOIN gold.fact_activities act ON act.opportunity_id = o.opportunity_id
WHERE o.stage IN ('won', 'lost')
GROUP BY o.opportunity_id, o.amount, o.stage, a.industry, a.country, a.annual_revenue, a.employees
"""

df = pd.read_sql(query, conn)
df["target"] = (df["stage"] == "won").astype(int)

print(f"Total de oportunidades cerradas: {len(df)}")
print(f"Distribución de la variable objetivo:\n{df['target'].value_counts(normalize=True)}")
df.head()


Total de oportunidades cerradas: 779
Distribución de la variable objetivo:
target
1    0.61104
0    0.38896
Name: proportion, dtype: float64


/tmp/ipykernel_12596/2320578102.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,opportunity_id,amount,stage,industry,country,annual_revenue,employees,activity_count,target
0,OPP-0001445,26545.75,won,services,AR,147034.52,94,4,1
1,OPP-0000731,25434.69,won,retail,MX,1935294.03,82,2,1
2,OPP-0002069,13776.83,won,education,BR,341200.96,42,1,1
3,OPP-0002552,10234.19,lost,manufacturing,CL,747802.22,190,4,0
4,OPP-0001172,3377.28,won,retail,CO,661855.87,13,6,1


## 2. Baseline ingenuo -- el punto de comparación obligatorio

Antes de entrenar cualquier modelo, hay que saber qué tan bien le iría a
un modelo "tonto" que siempre predice la clase mayoritaria. Si el win rate
real es ~61%, un modelo que *siempre* dice "ganada" ya acertaría 61% de las
veces sin aprender nada -- cualquier modelo real tiene que superar ese
número para justificar su existencia.


In [3]:
baseline_accuracy = df["target"].value_counts(normalize=True).max()
print(f"Baseline (predecir siempre la clase mayoritaria): {baseline_accuracy:.1%}")


Baseline (predecir siempre la clase mayoritaria): 61.1%


## 3. Entrenar el modelo

Features numéricas: `amount`, `annual_revenue`, `employees`, `activity_count`.
Features categóricas: `industry`, `country` (codificadas con one-hot).

Split 75/25 train/test, con `random_state` fijo para que el resultado sea
reproducible.


In [4]:
FEATURES_NUM = ["amount", "annual_revenue", "employees", "activity_count"]
FEATURES_CAT = ["industry", "country"]

X = df[FEATURES_NUM + FEATURES_CAT]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), FEATURES_NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), FEATURES_CAT),
])

model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)
print("Modelo entrenado.")


Modelo entrenado.


## 4. Evaluar -- comparando siempre contra el baseline

Un accuracy alto por sí solo no dice nada si no se compara contra el
baseline ingenuo de arriba.


In [5]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f"Accuracy del modelo:     {acc:.1%}")
print(f"Baseline (clase mayoritaria): {baseline_accuracy:.1%}")
print(f"Diferencia vs. baseline:  {(acc - baseline_accuracy)*100:+.1f} puntos porcentuales")
print(f"\nAUC-ROC: {auc:.3f}  (0.5 = no mejor que azar, 1.0 = perfecto)")
print(f"\n{classification_report(y_test, y_pred, target_names=['lost', 'won'])}")


Accuracy del modelo:     61.5%
Baseline (clase mayoritaria): 61.1%
Diferencia vs. baseline:  +0.4 puntos porcentuales

AUC-ROC: 0.476  (0.5 = no mejor que azar, 1.0 = perfecto)

              precision    recall  f1-score   support

        lost       1.00      0.01      0.03        76
         won       0.61      1.00      0.76       119

    accuracy                           0.62       195
   macro avg       0.81      0.51      0.39       195
weighted avg       0.76      0.62      0.47       195



## 5. ¿Qué variables pesan más? -- conexión directa con el hallazgo de discovery

Los coeficientes de la regresión logística indican qué tanto empuja cada
variable hacia "ganada" (positivo) o "perdida" (negativo), ya en la misma
escala gracias al `StandardScaler`.


In [6]:
feature_names = (
    FEATURES_NUM +
    list(model.named_steps["preprocess"].named_transformers_["cat"].get_feature_names_out(FEATURES_CAT))
)
coefs = model.named_steps["classifier"].coef_[0]

importance = pd.DataFrame({"feature": feature_names, "coef": coefs}).sort_values("coef", key=abs, ascending=False)
importance.head(15)


,feature,coef
6,industry_finance,0.433141
13,country_BR,-0.343950
18,country_PE,0.233889
7,industry_health,-0.164317
1,annual_revenue,0.138268
9,industry_retail,-0.107336
3,activity_count,-0.100012
10,industry_services,-0.093642
16,country_ES,0.076158
8,industry_manufacturing,0.070009


**Hallazgo esperado, a confirmar con el resultado real de arriba:** dado
que en el análisis de discovery el promedio de actividades era casi
idéntico entre `won` y `lost` (~3.4 vs ~3.6), es razonable esperar que
`activity_count` tenga un coeficiente cercano a cero -- es decir, que el
modelo confirme estadísticamente lo que ya habíamos visto con el promedio
simple: la cantidad de actividades no es un buen predictor del resultado.

_(completa aquí con el coeficiente real de activity_count una vez corras
la celda de arriba, y si el resultado confirma o contradice la hipótesis)_


## 6. Conclusión y limitaciones -- para ser honesto en la presentación

_(completar después de correr el notebook con los datos reales)_

- Accuracy del modelo vs. baseline:
- ¿Vale la pena el modelo, o el baseline ingenuo ya es casi tan bueno?
- Variable(s) más influyente(s):
- **Limitaciones a declarar:** modelo simple (regresión logística, sin
  tuning de hiperparámetros), dataset relativamente pequeño para estándares
  de ML, no se validó con cross-validation ni con un conjunto de datos
  fuera de tiempo (out-of-time validation) -- esto es un prototipo/prueba
  de concepto, no un modelo listo para producción.
